# Kiểm thử package `eda_v2` trên `data/mock_transactions.csv`

Năm hàm, mỗi hàm trả về `(figure, bảng số liệu)`; notebook xem figure bằng `show(fig)` và in bảng ra để đối chiếu.

| Hàm | Dùng cho |
|---|---|
| `plot_category_distribution` | cột phân loại: top K, nhóm Other, count, percentage |
| `plot_numeric_comparison` | so sánh phân phối cột số giữa các nhóm, tuỳ chọn ẩn outlier |
| `plot_numeric_distribution` | histogram, mean, median, percentile |
| `plot_entity_coverage` | hoạt động của entity theo thời gian |
| `plot_temporal_distribution` | aggregate, resample theo thời gian |



In [ ]:
import sys
from io import BytesIO
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # không phụ thuộc backend của kernel; figure được hiện bằng show(fig)
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
if not (repo_root / "eda_v2").exists():  # đang chạy từ notebooks/
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from eda_v2 import (plot_category_distribution, plot_numeric_comparison, plot_numeric_distribution,
                    plot_entity_coverage, plot_temporal_distribution)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


def show(fig):
    # Hiện figure dưới dạng ảnh PNG (chạy đúng với mọi backend), rồi đóng figure
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    display(Image(buf.getvalue()))



In [ ]:
df = pd.read_csv(repo_root / "data" / "mock_transactions.csv", parse_dates=["order_date"])
print(f"{df.shape[0]:,} dòng x {df.shape[1]} cột")
display(df.head())
display(df.dtypes.to_frame("dtype").T)



---
## 1. `plot_category_distribution`

Mặc định `top_k=10`, các nhóm ít gặp hơn được gộp vào `Other`.



In [ ]:
fig, table = plot_category_distribution(df, "payment_method")
show(fig)
display(table)



Ép `top_k=2` để thấy nhóm `Other` xuất hiện; tổng `percentage` luôn bằng 100.

In [ ]:
fig, table = plot_category_distribution(df, "shipping_region", top_k=2, other_label="Khác", figsize=(10, 4))
show(fig)
display(table)
assert abs(table["percentage"].sum() - 100) < 1e-9
assert table["count"].sum() == df["shipping_region"].notna().sum()



`top_k=None` giữ mọi nhóm, không gộp.

In [ ]:
fig, table = plot_category_distribution(df, "order_status", top_k=None)
show(fig)
assert "Other" not in table["category"].tolist()



---
## 2. `plot_numeric_comparison`

So sánh `total_amount` giữa các phương thức thanh toán; bảng là `describe()` theo nhóm.



In [ ]:
fig, table = plot_numeric_comparison(df, "total_amount", "payment_method")
show(fig)
display(table)



`hide_outliers=True` bỏ các điểm outlier khỏi chart (mặc định `False` = hiện); bảng `describe()` vẫn tính trên toàn bộ dữ liệu.

In [ ]:
fig, table = plot_numeric_comparison(df, "total_amount", "payment_method", hide_outliers=True)
show(fig)



In [ ]:
fig, table = plot_numeric_comparison(df, "delivery_days", "shipping_region", top_k=2)
show(fig)
display(table)
assert len(table) <= 2



---
## 3. `plot_numeric_distribution`

Histogram kèm mean (đỏ), median (xanh), percentile (xám). `total_amount` lệch phải, nên mean > median.



In [ ]:
fig, stats = plot_numeric_distribution(df, "total_amount")
show(fig)
display(stats)



In [ ]:
fig, stats = plot_numeric_distribution(df, "customer_age", bins=15, percentiles=(10, 50, 90))
show(fig)
display(stats)



`metric_with_inf` chứa ±inf: hàm phải bỏ chúng trước khi tính, nếu không mean/median sẽ là inf/nan.

In [ ]:
fig, stats = plot_numeric_distribution(df, "metric_with_inf", percentiles=())
show(fig)
display(stats)
assert stats.notna().all() and stats.abs().lt(float("inf")).all()



---
## 4. `plot_entity_coverage`

Heatmap số dòng của mỗi `customer_id` theo kỳ. Hàng = entity (sắp theo mức hoạt động), cột = kỳ.



In [ ]:
fig, table = plot_entity_coverage(df, "customer_id", "order_date", freq="M", top_k=30)
show(fig)
display(table.iloc[:5, :6])
assert table.shape[0] <= 30



Kỳ theo tuần và chỉ top 10 entity.

In [ ]:
fig, table = plot_entity_coverage(df, "customer_id", "order_date", freq="W", top_k=10, figsize=(16, 5))
show(fig)
print(table.shape)



---
## 5. `plot_temporal_distribution`

Không truyền `value_column` thì đếm số dòng; có `value_column` thì aggregate theo `agg`. Lưu ý resample theo tháng dùng `freq="ME"` (pandas >= 2.2).



In [ ]:
fig, series = plot_temporal_distribution(df, "order_date", freq="W")
show(fig)
display(series.head())
assert series.sum() == df["order_date"].notna().sum()



In [ ]:
fig, series = plot_temporal_distribution(df, "order_date", "total_amount", freq="ME", agg="sum")
show(fig)
display(series.head())



In [ ]:
fig, series = plot_temporal_distribution(df, "order_date", "total_amount", freq="ME", agg="mean")
show(fig)
display(series.head())



---
## 6. Không sửa dữ liệu đầu vào

Sau mọi lần gọi ở trên, `df` phải giữ nguyên.



In [ ]:
df_after = pd.read_csv(repo_root / "data" / "mock_transactions.csv", parse_dates=["order_date"])
pd.testing.assert_frame_equal(df, df_after)
print("df không bị sửa.")

